In [4]:
# pip install mlflow

import mlflow
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame


In [ ]:
df.target.value_counts()

# 1 = malignant
# 0 = benign

target
1    357
0    212
Name: count, dtype: int64

In [7]:
X = df.drop('target', axis=1)  # everything except the label
y = df['target']               # the label column

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% of data goes to test, 80% to train
    random_state=42,  # fixed seed → same split every time → reproducible
    stratify=y        # ensures class ratio is same in train and test
)

In [12]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Training samples : {X_train_sc.shape[0]}')
print(f'Test samples     : {X_test_sc.shape[0]}')
print(f'Features         : {X_train_sc.shape[1]}')
print(f'Class balance (train): {pd.Series(y_train).value_counts().to_dict()}')

Training samples : 455
Test samples     : 114
Features         : 30
Class balance (train): {1: 285, 0: 170}


In [ ]:
# ──────────────────────────────────────────────────────────
# TRACKING URI — where MLflow stores all data
#
# Option A (what we use): sqlite:///mlflow.db
#   → stores everything in a local SQLite file called mlflow.db
#   → no server needed, works offline, great for learning
#
# Option B (production): http://my-mlflow-server:5000
#   → a shared remote server that the whole team points to

In [13]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')

# ──────────────────────────────────────────────────────────
# EXPERIMENT — a logical container that groups related runs
# Think: one experiment per model type or per business problem
#
# If the experiment does not exist yet, MLflow creates it.
# If it already exists, it just activates it.
# ──────────────────────────────────────────────────────────

mlflow.search_experiments('cnacer_prediction-week1')


2026/06/23 11:22:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/23 11:22:04 INFO mlflow.store.db.utils: Updating database tables


[]

In [17]:
with mlflow.start_run(run_name='Logistic Regression'):
    lr = LogisticRegression()
    lr.fit(X_train_sc, y_train)

    y_pred = lr.predict(X_test_sc)
    accuracy = accuracy_score(y_test, y_pred)

    mlflow.log_params({
        'model':        'LogisticRegression',
        'solver':       'lbfgs',
        'scaler':       'StandardScaler',
        'random_state':  42,
    })
    mlflow.log_metric('accuracy', accuracy)

    mlflow.sklearn.log_model(lr, 'model', registered_model_name='Breast Cancer Classifier')

2026/06/23 11:27:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'Breast Cancer Classifier' already exists. Creating a new version of this model...
Created version '3' of model 'Breast Cancer Classifier'.
